In [38]:
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [39]:
import torch

In [40]:
with open('input.txt', 'r', encoding='utf-8') as file:
    text = file.read()

print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [54]:
print(f"dataset length:")
print(f"{len(text)} chars")
print(f"{len(text.split(' '))} words")

dataset length:
1115394 chars
169893 words


In [42]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


Tokenizer

In [43]:
stoi = {char:i for i,char in enumerate(chars)}
itos = {i:char for char,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s] # string -> list[int]
decode = lambda l: ''.join(itos[i] for i in l) # list[int] -> string

print(decode(encode("Naman")))

Naman


In [45]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


Train and Validation Data Sets

In [47]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [57]:
block_size = 8
print(train_data[:block_size + 1])
print(decode(train_data[:block_size + 1].tolist()))

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])
First Cit


In [63]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
print(x)
print(y)

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"context: {context} --> target: {target}")

tensor([18, 47, 56, 57, 58,  1, 15, 47])
tensor([47, 56, 57, 58,  1, 15, 47, 58])
context: tensor([18]) --> target: 47
context: tensor([18, 47]) --> target: 56
context: tensor([18, 47, 56]) --> target: 57
context: tensor([18, 47, 56, 57]) --> target: 58
context: tensor([18, 47, 56, 57, 58]) --> target: 1
context: tensor([18, 47, 56, 57, 58,  1]) --> target: 15
context: tensor([18, 47, 56, 57, 58,  1, 15]) --> target: 47
context: tensor([18, 47, 56, 57, 58,  1, 15, 47]) --> target: 58


In [ ]:
batch_size = 4 # batches of blocks to process in parallel
block_size = 8 # size of blocks to train on

def get_batch(split):
    match split:
        case 'train':
            data = train_data
        case 'validation':
            data = val_data
    
    index = torch.randint(len(data) - block_size, )